In [2]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

## Variant diff

In [3]:
gb_vars = pl.read_parquet('/home/dnanexus/data_dir/genebass1e6_394genes_10kb_variants.parquet')
gb_vars

id,chrom,pos,ref,alt,region,gene_name
str,str,i32,str,str,str,str
"""chr2:169197167:T:G""","""chr2""",169197167,"""T""","""G""","""ENSG00000081479""","""LRP2"""
"""chr2:169204292:A:T""","""chr2""",169204292,"""A""","""T""","""ENSG00000081479""","""LRP2"""
"""chr2:169030531:C:T""","""chr2""",169030531,"""C""","""T""","""ENSG00000073734""","""ABCB11"""
"""chr2:168910825:T:G""","""chr2""",168910825,"""T""","""G""","""ENSG00000073734""","""ABCB11"""
"""chr2:168969544:T:C""","""chr2""",168969544,"""T""","""C""","""ENSG00000073734""","""ABCB11"""
…,…,…,…,…,…,…
"""chr6:30150733:A:C""","""chr6""",30150733,"""A""","""C""","""ENSG00000204614""","""TRIM40"""
"""chr15:34974609:ACTTCT:A""","""chr15""",34974609,"""ACTTCT""","""A""","""ENSG00000198146""","""ZNF770"""
"""chr9:132358570:C:A""","""chr9""",132358570,"""C""","""A""","""ENSG00000107290""","""SETX"""


In [4]:
olink_genes = pl.read_csv('/home/dnanexus/data_dir/olink_genebass1e6_371genes.tsv', separator='\t')['region'].to_list()

olink_var_ids = pl.scan_parquet(f"/home/dnanexus/data_dir/genebass1e6_genes_10kb_allvars_annotated_250925.parquet").filter(pl.col('region').is_in(olink_genes)).select(gb_vars.columns).collect(engine='streaming')
# olink_var_ids.write_parquet('/home/dnanexus/data_dir/olink_371genes_10kb_variants.parquet')
olink_var_ids

id,chrom,pos,ref,alt,region,gene_name
str,str,i32,str,str,str,str
"""chr2:169197167:T:G""","""chr2""",169197167,"""T""","""G""","""ENSG00000081479""","""LRP2"""
"""chr2:169204292:A:T""","""chr2""",169204292,"""A""","""T""","""ENSG00000081479""","""LRP2"""
"""chr2:169175752:G:A""","""chr2""",169175752,"""G""","""A""","""ENSG00000081479""","""LRP2"""
"""chr2:165362262:G:T""","""chr2""",165362262,"""G""","""T""","""ENSG00000136531""","""SCN2A"""
"""chr2:165386841:G:T""","""chr2""",165386841,"""G""","""T""","""ENSG00000136531""","""SCN2A"""
…,…,…,…,…,…,…
"""chr11:10328313:TC:T""","""chr11""",10328313,"""TC""","""T""","""ENSG00000133805""","""AMPD3"""
"""chr3:112330970:C:G""","""chr3""",112330970,"""C""","""G""","""ENSG00000091972""","""CD200"""
"""chr1:156890114:C:T""","""chr1""",156890114,"""C""","""T""","""ENSG00000187800""","""PEAR1"""


In [ ]:
# olink_var_ids.join(gb_vars, on=gb_vars.columns, how='anti')#.write_parquet('/home/dnanexus/data_dir/olink371genes_genebass394genes_variants_diff.parquet')

#.write_parquet('/home/dnanexus/data_dir/olink371genes_genebass394genes_variants_union.parquet')

id,chrom,pos,ref,alt,region,gene_name
str,str,i32,str,str,str,str
"""chr2:165362262:G:T""","""chr2""",165362262,"""G""","""T""","""ENSG00000136531""","""SCN2A"""
"""chr2:165386841:G:T""","""chr2""",165386841,"""G""","""T""","""ENSG00000136531""","""SCN2A"""
"""chr2:165365630:A:G""","""chr2""",165365630,"""A""","""G""","""ENSG00000136531""","""SCN2A"""
"""chr2:165367163:T:C""","""chr2""",165367163,"""T""","""C""","""ENSG00000136531""","""SCN2A"""
"""chr2:165383236:T:G""","""chr2""",165383236,"""T""","""G""","""ENSG00000136531""","""SCN2A"""
…,…,…,…,…,…,…
"""chr20:54166734:TA:T""","""chr20""",54166734,"""TA""","""T""","""ENSG00000019186""","""CYP24A1"""
"""chr11:10328313:TC:T""","""chr11""",10328313,"""TC""","""T""","""ENSG00000133805""","""AMPD3"""
"""chr3:112330970:C:G""","""chr3""",112330970,"""C""","""G""","""ENSG00000091972""","""CD200"""


In [8]:
gb_olink_vars = pl.concat([gb_vars.join(olink_var_ids, on=gb_vars.columns, how='anti'), olink_var_ids.join(gb_vars, on=gb_vars.columns, how='anti'), gb_vars.join(olink_var_ids, on=gb_vars.columns, how='inner')])

gb_olink_vars

id,chrom,pos,ref,alt,region,gene_name
str,str,i32,str,str,str,str
"""chr2:169030531:C:T""","""chr2""",169030531,"""C""","""T""","""ENSG00000073734""","""ABCB11"""
"""chr2:168910825:T:G""","""chr2""",168910825,"""T""","""G""","""ENSG00000073734""","""ABCB11"""
"""chr2:168969544:T:C""","""chr2""",168969544,"""T""","""C""","""ENSG00000073734""","""ABCB11"""
"""chr2:169007174:C:T""","""chr2""",169007174,"""C""","""T""","""ENSG00000073734""","""ABCB11"""
"""chr2:169018732:A:G""","""chr2""",169018732,"""A""","""G""","""ENSG00000073734""","""ABCB11"""
…,…,…,…,…,…,…
"""chr15:98793613:CAG:C""","""chr15""",98793613,"""CAG""","""C""","""ENSG00000140443""","""IGF1R"""
"""chr1:156892240:C:T""","""chr1""",156892240,"""C""","""T""","""ENSG00000187800""","""PEAR1"""
"""chr1:156917808:T:C""","""chr1""",156917808,"""T""","""C""","""ENSG00000187800""","""PEAR1"""


In [ ]:
# gb_olink_vars.write_parquet('/home/dnanexus/data_dir/olink371genes_genebass394genes_variants_union.parquet')

In [3]:
all_vars = pl.scan_parquet(f"/home/dnanexus/data_dir/genebass1e6_genes_10kb_allvars_annotated_250925.parquet")
all_vars.head().collect()

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,sngl10000bp_is_nan,ensembleregulatoryfeature_is_nan,dbscsnv-ada_score_is_nan,dbscsnv-rf_score_is_nan,remapoverlaptf_is_nan,remapoverlapcl_is_nan,esmscoremissense_is_nan,esmscoreinframe_is_nan,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32
"""chr2""",134144266,"""C""","""T""","""chr2:134144266:C:T""",11108971,"""ENSG00000152127""",0,0.0,0.955524,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,134119982,"""+""",334640,"""MGAT5""",24284,-0.011144,-0.012463,0.0,0.0,0.0
"""chr2""",134192587,"""C""","""T""","""chr2:134192587:C:T""",11127058,"""ENSG00000152127""",0,0.0,0.25085,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,134119982,"""+""",334640,"""MGAT5""",72605,-0.00487,-0.006189,0.0,0.0,0.0
"""chr2""",134389573,"""T""","""C""","""chr2:134389573:T:C""",11199699,"""ENSG00000152127""",0,0.0,1.25828,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,134119982,"""+""",334640,"""MGAT5""",269591,-0.011792,-0.013111,0.0,0.0,0.0
"""chr2""",127525503,"""G""","""A""","""chr2:127525503:G:A""",11084322,"""ENSG00000163166""",0,0.0,0.395322,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,127526886,"""-""",90681,"""IWS1""",1383,0.003182,-0.006438,0.0,0.0,0.0
"""chr2""",134155422,"""C""","""T""","""chr2:134155422:C:T""",11112693,"""ENSG00000152127""",0,0.0,0.240848,0.0,0.04,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,134119982,"""+""",334640,"""MGAT5""",35440,-0.00487,-0.006189,0.0,0.0,0.0


In [ ]:
gb_olink_vars = pl.scan_parquet('/home/dnanexus/data_dir/genebass394genes_olink371genes_variants_union.parquet')
id_cols = gb_olink_vars.collect_schema().names()
print(id_cols)
gb_olink_vars.head().collect()

['id', 'chrom', 'pos', 'ref', 'alt', 'region', 'gene_name']


id,chrom,pos,ref,alt,region,gene_name
str,str,i32,str,str,str,str
"""chr2:169030531:C:T""","""chr2""",169030531,"""C""","""T""","""ENSG00000073734""","""ABCB11"""
"""chr2:168910825:T:G""","""chr2""",168910825,"""T""","""G""","""ENSG00000073734""","""ABCB11"""
"""chr2:168969544:T:C""","""chr2""",168969544,"""T""","""C""","""ENSG00000073734""","""ABCB11"""
"""chr2:169007174:C:T""","""chr2""",169007174,"""C""","""T""","""ENSG00000073734""","""ABCB11"""
"""chr2:169018732:A:G""","""chr2""",169018732,"""A""","""G""","""ENSG00000073734""","""ABCB11"""


In [7]:
gb_olink_vars_anno = all_vars.join(gb_olink_vars, on=['id', 'region'], how='semi').collect(engine='streaming')
gb_olink_vars_anno

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,sngl10000bp_is_nan,ensembleregulatoryfeature_is_nan,dbscsnv-ada_score_is_nan,dbscsnv-rf_score_is_nan,remapoverlaptf_is_nan,remapoverlapcl_is_nan,esmscoremissense_is_nan,esmscoreinframe_is_nan,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.0,0.095799,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,132354986,""

In [8]:
gb_olink_vars_anno['region'].value_counts(sort=True)

region,count
str,u64
"""ENSG00000113448""",608129
"""ENSG00000187323""",533458
"""ENSG00000178568""",484385
"""ENSG00000169855""",480249
"""ENSG00000171587""",352544
…,…
"""ENSG00000169469""",4699
"""ENSG00000132703""",4693
"""ENSG00000169877""",4172


In [ ]:
# gb_olink_vars_anno.write_parquet('/home/dnanexus/data_dir/genebass394genes_olink371genes_variants_union_annotated.parquet')

## coding variants

In [2]:
anngeno_path = '/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
# olink_path = '/home/dnanexus/data_dir/olink/protrider_lite_output/log2fc.csv'
olink_path = "/home/dnanexus/data_dir/olink/olink_corrected_rint_90_pcs.parquet"

In [3]:
sample_ids = zarr.open(f'{anngeno_path}/zarr_store/samples', mode='r')[:]
var_ids = pl.read_parquet(f'{anngeno_path}/variant_metadata.parquet', columns=['id'])['id'].to_numpy()
geno = zarr.open(f'{anngeno_path}/zarr_store/genotypes', mode='r')

eur_samples = pl.read_csv(eur_samples_path).rename({'eid': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
)['individual'].to_list()

/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


In [4]:
olink_df = pl.read_parquet(olink_path).rename({'sample': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
).filter(
    pl.col("individual").is_in(eur_samples)
).fill_nan(None)

unique_phenotypes = [pheno for pheno in olink_df.columns if pheno != 'individual']

olink_melt = (
    olink_df.unpivot(
        index=['individual'],
        on=unique_phenotypes,
        variable_name='phenotype',
        value_name='pheno_value',
    )
    .with_columns(
        phenotype = pl.col('phenotype') + '_olink'
    )
)

olink_melt

individual,phenotype,pheno_value
str,str,f64
"""1000107""","""ENSG00000120053_olink""",-0.592022
"""1001193""","""ENSG00000120053_olink""",0.061036
"""1001694""","""ENSG00000120053_olink""",0.228713
"""1001715""","""ENSG00000120053_olink""",0.674837
"""1002486""","""ENSG00000120053_olink""",1.153924
…,…,…
"""6019565""","""ENSG00000066468_olink""",0.72708
"""6019855""","""ENSG00000066468_olink""",-0.327369
"""6021528""","""ENSG00000066468_olink""",0.991073


In [5]:
# Optimized approach to find indices of olink_samples in sample_ids
# Step A: Create a lookup dictionary mapping each ID in the large array to its original index. This takes O(N) time, where N is the size of sample_ids.
olink_sids = olink_df['individual'].to_numpy()
sample_id_to_index = {sid: i for i, sid in enumerate(sample_ids)}

# Step B: Iterate through the smaller array and find the index for each element if it exists in our lookup dictionary. This takes O(M) time, where M is the size of olink_sids.
found_indices = []
for sid in olink_sids:
    if sid in sample_id_to_index:
        found_indices.append(sample_id_to_index[sid])
print(f"Found {len(found_indices)} matching IDs.")

# Step C: Convert the list of indices to a NumPy array and sort it.
# Sorting ensures the output is identical to the original np.where approach, which returns indices in ascending order.
olink_indices = np.sort(found_indices)
olink_indices

Found 40398 matching IDs.


array([    17,     23,     45, ..., 490517, 490532, 490535],
      shape=(40398,))

In [8]:
sample_ids[olink_indices]

array(['5102203', '3741733', '4072771', ..., '5141119', '5145210',
       '5733954'], shape=(40398,), dtype=StringDType())

In [9]:
len(set(olink_lazy['individual'].to_list()).intersection(set(sample_ids[olink_indices])))

NameError: name 'olink_lazy' is not defined

In [10]:
@numba.njit(parallel=True, fastmath=True)
def _fast_clip_and_sum_allels(arr):
    # Get the shape of the input array
    # Using specific dimensions for clarity with this problem
    n_samples, n_variants, _ = arr.shape
    
    # The sum of two positive int8s can be up to 254. 
    # An int16 is a safe and fast output type.
    output = np.empty((n_samples, n_variants), dtype=np.int8)
    
    # Numba's prange enables automatic parallelization across all your CPU cores
    for i in numba.prange(n_samples):
        for j in range(n_variants):
            # Read two values, perform logic, write one value.
            # This is the "fused" operation.
            val1 = arr[i, j, 0]
            val2 = arr[i, j, 1]
            
            s = 0
            # Since input is int8, this check is faster than max(0, val)
            if val1 > 0:
                s += val1
            if val2 > 0:
                s += val2
            
            output[i, j] = s
            
    return output

def process_genotype_chunk(
    geno: np.array, 
    var_ids: np.array, 
    sample_list: np.array,
    melted_pheno_df: pl.LazyFrame,
    homozygous: bool = False,
    debug: bool = False,
) -> pl.LazyFrame:
    """
    Extract genotypes for a specific gene and return as lazy DataFrame
    """
    geno_clipped = _fast_clip_and_sum_allels(geno)
    # Find heterozygous genotypes (genotype == 1)
    rows, cols = np.where(geno_clipped == 1)
    geno_melt = pl.DataFrame({
        'id': var_ids[rows],
        'individual': sample_list[cols],
        'genotype': 1
    })
    
    # Find homozygous genotypes (genotype == 2)
    if homozygous:
        rows, cols = np.where(geno_clipped == 2)
        hom = pl.DataFrame({
            'id': var_ids[rows],
            'individual': sample_list[cols],
            'genotype': 2
        })
        geno_melt = pl.concat([geno_melt, hom])

    var_pheno_df = geno_melt.lazy().join(melted_pheno_df, on='individual', how='left')
    if debug:
        # Return intermediate dataframe if debugging
        return var_pheno_df
    
    var_pheno_df = var_pheno_df.group_by(
           ['id', 'phenotype']
           ).agg([
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().cast(pl.Float32).alias('mean_pheno_value'),
                pl.col('pheno_value').std().cast(pl.Float32).alias('std_pheno_value'),
            ]).drop_nulls(subset=['mean_pheno_value'])
    return var_pheno_df

In [16]:
output_dir = "/home/dnanexus/data_dir/var_pheno_EUR_chunks"
chunk_size = 10_000

for chunk_num in tqdm(range(var_ids.shape[0]//chunk_size + 1)):
    tmp = process_genotype_chunk(
        geno=geno[chunk_num*chunk_size:(chunk_num+1)*chunk_size, olink_indices],
        var_ids=var_ids[chunk_num*chunk_size:(chunk_num+1)*chunk_size],
        sample_list=sample_ids[olink_indices],
        melted_pheno_df=olink_melt.lazy(),
        homozygous=False,
    ).collect()
    break

tmp

  0%|          | 0/183 [00:17<?, ?it/s]


id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:3467169:C:G""","""ENSG00000081800_olink""",1,-0.363691,null
"""chr1:2229364:A:G""","""ENSG00000135218_olink""",1,0.408683,null
"""chr1:8967768:C:T""","""ENSG00000082074_olink""",7,0.198808,1.265288
"""chr1:6213135:C:T""","""ENSG00000065361_olink""",4,0.381372,1.242684
"""chr1:3480477:GGA:G""","""ENSG00000124678_olink""",1,1.095063,null
…,…,…,…,…
"""chr1:2519307:T:C""","""ENSG00000168769_olink""",1,-0.429507,null
"""chr1:1374025:T:C""","""ENSG00000019991_olink""",1496,-0.004606,1.019444
"""chr1:10105685:T:C""","""ENSG00000120889_olink""",1,-0.988386,null


In [17]:
tmp['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:2229364:A:G""",371
"""chr1:8967768:C:T""",371
"""chr1:6213135:C:T""",371
"""chr1:3480477:GGA:G""",371
"""chr1:2519857:C:T""",371
…,…
"""chr1:11059839:C:A""",123
"""chr1:2518220:C:T""",99
"""chr1:11056741:G:C""",63


## Debug

In [18]:
geno_clipped = _fast_clip_and_sum_allels(geno[:10_000, olink_indices])
geno_clipped

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(10000, 40398), dtype=int8)

In [25]:
sum(geno_clipped.sum(axis=1) > 0)

np.int64(2338)

In [19]:
rows, cols = np.where(geno_clipped == 1)

In [20]:
sample_list=sample_ids[olink_indices]
geno_melt = pl.DataFrame({
        'id': var_ids[rows],
        'individual': sample_list[cols],
        'genotype': 1
    })

geno_melt

id,individual,genotype
str,str,i32
"""chr1:1373805:T:C""","""3622589""",1
"""chr1:1373805:T:C""","""3540179""",1
"""chr1:1373805:T:C""","""3991045""",1
"""chr1:1373805:T:C""","""3130113""",1
"""chr1:1373805:T:C""","""4471351""",1
…,…,…
"""chr1:11231009:G:A""","""2259790""",1
"""chr1:11231013:G:A""","""2141582""",1
"""chr1:11231040:T:C""","""2938277""",1


In [21]:
var_pheno_df = geno_melt.join(olink_melt, on='individual', how='left')
var_pheno_df

id,individual,genotype,phenotype,pheno_value
str,str,i32,str,f64
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000120053_olink""",-1.273965
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000140443_olink""",1.564473
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000084674_olink""",-0.164796
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000015475_olink""",-1.803142
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000117748_olink""",-0.09892
…,…,…,…,…
"""chr1:11231048:G:A""","""3611469""",1,"""ENSG00000124678_olink""",0.442166
"""chr1:11231048:G:A""","""3611469""",1,"""ENSG00000109062_olink""",1.012175
"""chr1:11231048:G:A""","""3611469""",1,"""ENSG00000167186_olink""",0.72797


In [23]:
var_pheno_df['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:8949385:A:C""",7447825
"""chr1:8949347:C:T""",7087584
"""chr1:2512975:G:A""",7044548
"""chr1:6218354:A:G""",6735134
"""chr1:8957145:A:G""",6316646
…,…
"""chr1:11231009:G:A""",371
"""chr1:11231013:G:A""",371
"""chr1:11231040:T:C""",371


In [26]:
vm = var_pheno_df.group_by(
           ['id', 'phenotype']
           ).agg([
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().cast(pl.Float32).alias('mean_pheno_value'),
                pl.col('pheno_value').std().cast(pl.Float32).alias('std_pheno_value'),
            ])#.drop_nulls(subset=['mean_pheno_value'])
vm

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:11204686:C:A""","""ENSG00000055955_olink""",1,-0.891984,null
"""chr1:3469480:G:A""","""ENSG00000188211_olink""",1,-1.286588,null
"""chr1:2306115:C:T""","""ENSG00000019186_olink""",2,-1.203303,0.789745
"""chr1:2512947:C:T""","""ENSG00000081800_olink""",3,-0.992049,0.70909
"""chr1:3478504:T:C""","""ENSG00000169194_olink""",1,-0.871757,null
…,…,…,…,…
"""chr1:3477964:C:T""","""ENSG00000089902_olink""",1,-0.079269,null
"""chr1:3463171:G:A""","""ENSG00000165471_olink""",2,0.296308,0.593581
"""chr1:1374732:G:A""","""ENSG00000132703_olink""",1,null,null


In [29]:
vm.drop_nulls(subset=['mean_pheno_value'])['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:3469480:G:A""",371
"""chr1:2306115:C:T""",371
"""chr1:2512947:C:T""",371
"""chr1:8945887:A:G""",371
"""chr1:1374375:C:T""",371
…,…
"""chr1:11059839:C:A""",123
"""chr1:2518220:C:T""",99
"""chr1:3732885:G:A""",63


## WGS debug

In [4]:
anngeno_path = '/home/dnanexus/data_dir/genebass_1e6.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
# olink_path = '/home/dnanexus/data_dir/olink/protrider_lite_output/log2fc.csv'
olink_path = "/home/dnanexus/data_dir/olink/olink_corrected_rint_90_pcs.parquet"

In [5]:
sample_ids = zarr.open(f'{anngeno_path}/zarr_store/samples', mode='r')[:]
var_ids = pl.read_parquet(f'{anngeno_path}/variant_metadata.parquet', columns=['id'])['id'].to_numpy()
geno = zarr.open(f'{anngeno_path}/zarr_store/genotypes', mode='r')

eur_samples = pl.read_csv(eur_samples_path).rename({'eid': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
)['individual'].to_list()


In [6]:
olink_df = pl.read_parquet(olink_path).rename({'sample': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
).filter(
    pl.col("individual").is_in(eur_samples)
).fill_nan(None)

unique_phenotypes = [pheno for pheno in olink_df.columns if pheno != 'individual']

olink_melt = (
    olink_df.unpivot(
        index=['individual'],
        on=unique_phenotypes,
        variable_name='phenotype',
        value_name='pheno_value',
    )
    .with_columns(
        phenotype = pl.col('phenotype') + '_olink'
    )
)

olink_melt

individual,phenotype,pheno_value
str,str,f64
"""1000107""","""ENSG00000120053_olink""",-0.592022
"""1001193""","""ENSG00000120053_olink""",0.061036
"""1001694""","""ENSG00000120053_olink""",0.228713
"""1001715""","""ENSG00000120053_olink""",0.674837
"""1002486""","""ENSG00000120053_olink""",1.153924
…,…,…
"""6019565""","""ENSG00000066468_olink""",0.72708
"""6019855""","""ENSG00000066468_olink""",-0.327369
"""6021528""","""ENSG00000066468_olink""",0.991073


In [9]:
region_list = [g[:-6] for g in olink_melt['phenotype'].unique().to_list()]
len(region_list)

371

In [11]:
# Optimized approach to find indices of olink_samples in sample_ids
# Step A: Create a lookup dictionary mapping each ID in the large array to its original index. This takes O(N) time, where N is the size of sample_ids.
olink_sids = olink_df['individual'].to_numpy()
sample_id_to_index = {sid: i for i, sid in enumerate(sample_ids)}

# Step B: Iterate through the smaller array and find the index for each element if it exists in our lookup dictionary. This takes O(M) time, where M is the size of olink_sids.
found_indices = []
for sid in olink_sids:
    if sid in sample_id_to_index:
        found_indices.append(sample_id_to_index[sid])
print(f"Found {len(found_indices)} matching IDs.")

# Step C: Convert the list of indices to a NumPy array and sort it.
# Sorting ensures the output is identical to the original np.where approach, which returns indices in ascending order.
olink_indices = np.sort(found_indices)

Found 40398 matching IDs.


In [12]:
found_indices

[192701,
 179743,
 234538,
 452916,
 221385,
 12142,
 366951,
 40441,
 357078,
 171701,
 362876,
 431476,
 402558,
 395145,
 148779,
 22357,
 376940,
 289660,
 568,
 365464,
 411572,
 210410,
 81605,
 171708,
 411518,
 452881,
 327493,
 351496,
 433356,
 92197,
 156509,
 146220,
 487659,
 355664,
 394012,
 359466,
 459929,
 17007,
 371537,
 252293,
 391108,
 98590,
 174864,
 124300,
 130239,
 149835,
 128247,
 171344,
 95055,
 179270,
 332008,
 17030,
 167350,
 69328,
 418672,
 374426,
 290813,
 427985,
 473786,
 359379,
 373226,
 240143,
 420990,
 462300,
 442224,
 34241,
 192832,
 471995,
 25845,
 419077,
 404354,
 310964,
 292464,
 128991,
 18174,
 11247,
 110055,
 177898,
 419401,
 200689,
 380549,
 435216,
 64634,
 370879,
 339062,
 190257,
 26373,
 19453,
 306444,
 229225,
 405742,
 478623,
 286172,
 6345,
 314175,
 81720,
 402953,
 483576,
 317441,
 64218,
 365629,
 233531,
 112074,
 278820,
 271200,
 6789,
 464404,
 67696,
 323078,
 77770,
 158331,
 218918,
 182294,
 464474,
 3

In [13]:
olink_indices

array([    17,     23,     45, ..., 490517, 490532, 490535],
      shape=(40398,))

In [14]:
gene_name = 'LDLR'
anno = pl.scan_parquet(f"/home/dnanexus/data_dir/genebass1e6_genes_10kb_allvars_annotated_250925.parquet").filter(pl.col('gene_name') == gene_name).select(['id']).unique().collect(engine='streaming')
anno

id
str
"""chr19:11091529:G:C"""
"""chr19:11136598:G:A"""
"""chr19:11122444:G:T"""
"""chr19:11095648:G:C"""
"""chr19:11099942:T:C"""
…
"""chr19:11104240:A:AAGG"""
"""chr19:11091225:CT:C"""
"""chr19:11097504:C:CAAAAACA"""


In [18]:
pl.scan_parquet(f"/home/dnanexus/data_dir/genebass1e6_genes_10kb_allvars_annotated_250925.parquet").filter(pl.col('region').is_in(unique_phenotypes)).select(['id']).unique().collect(engine='streaming')

id
str
"""chr2:105843356:G:C"""
"""chr2:105778273:T:G"""
"""chr2:53777089:C:A"""
"""chr2:120142458:G:A"""
"""chr2:157293678:C:T"""
…
"""chr2:212280225:A:C"""
"""chr2:212051994:G:A"""
"""chr2:212113335:A:C"""


In [24]:
anno_var_ids = np.concatenate((anno['id'].to_numpy(), ['chr19:11093738:G:A']))
anno_var_ids

array(['chr19:11093738:G:A', 'chr19:11110949:G:A', 'chr19:11137687:A:C',
       ..., 'chr19:11097297:CG:C', 'chr19:11126535:AG:A',
       'chr19:11093738:G:A'], shape=(23998,), dtype=object)

In [10]:
# Optimized approach to find indices of olink_samples in sample_ids
# Step A: Create a lookup dictionary mapping each ID in the large array to its original index. This takes O(N) time, where N is the size of sample_ids.
var_ids_to_index = {sid: i for i, sid in enumerate(var_ids)}

# Step B: Iterate through the smaller array and find the index for each element if it exists in our lookup dictionary. This takes O(M) time, where M is the size of olink_sids.
found_indices = []
for sid in anno_var_ids:
    if sid in var_ids_to_index:
        found_indices.append(var_ids_to_index[sid])
print(f"Found {len(found_indices)} matching IDs.")

# Step C: Convert the list of indices to a NumPy array and sort it.
# Sorting ensures the output is identical to the original np.where approach, which returns indices in ascending order.
indices_vars2keep = np.sort(found_indices)
indices_vars2keep

Found 23997 matching IDs.


array([81565377, 81565403, 81565404, ..., 81589396, 81589397, 81589398],
      shape=(23997,))

In [12]:
var_ids[indices_vars2keep]

array(['chr19:11084366:TTGGCTCATGTCTGTAATCCCAGCACGTTGGGAGGCTGAGGCGGGTGGATCACAAGGTCAGGAGATCCAGACCATCCTGGCTAACAAAGTGAAACCCCGTCTCTACTAAAAATACAAAAAATTAGGCTGGGCGTGG:T',
       'chr19:11084419:A:G', 'chr19:11084420:A:G', ...,
       'chr19:11138810:C:A', 'chr19:11138812:G:C', 'chr19:11138814:C:A'],
      shape=(23997,), dtype=object)

In [19]:
chunk_size = 1_000

for chunk_num in range(indices_vars2keep.shape[0] // chunk_size + 1):
    indices_vars2keep_chunk = indices_vars2keep[chunk_num*chunk_size:(chunk_num+1)*chunk_size]
    tmp_geno = geno.oindex[indices_vars2keep_chunk, olink_indices]
    tmp_vars = var_ids[indices_vars2keep_chunk]
    break

In [18]:
olink_indices

array([    17,     23,     45, ..., 490517, 490532, 490535],
      shape=(40398,))

In [20]:
indices_vars2keep_chunk

array([81565377, 81565403, 81565404, 81565405, 81565406, 81565407,
       81565408, 81565409, 81565410, 81565411, 81565412, 81565413,
       81565414, 81565415, 81565416, 81565417, 81565418, 81565419,
       81565420, 81565421, 81565422, 81565423, 81565424, 81565425,
       81565426, 81565427, 81565428, 81565429, 81565430, 81565431,
       81565432, 81565433, 81565434, 81565435, 81565436, 81565437,
       81565438, 81565439, 81565440, 81565441, 81565442, 81565443,
       81565444, 81565445, 81565446, 81565447, 81565448, 81565449,
       81565450, 81565451, 81565452, 81565453, 81565454, 81565455,
       81565456, 81565457, 81565458, 81565459, 81565460, 81565461,
       81565462, 81565463, 81565464, 81565465, 81565466, 81565467,
       81565468, 81565469, 81565470, 81565471, 81565472, 81565473,
       81565474, 81565475, 81565476, 81565477, 81565478, 81565479,
       81565480, 81565481, 81565482, 81565483, 81565484, 81565485,
       81565486, 81565487, 81565488, 81565489, 81565490, 81565

## Debug results

In [ ]:
output_dir = "/home/dnanexus/data_dir/olink_appv_chunks_EUR_smarter"

files = [f"{output_dir}/{filename}" for filename in os.listdir(output_dir) if filename.endswith(".parquet")]

# lazy_frames = [pl.scan_parquet(f) for f in files]
# combined = pl.concat(lazy_frames)

# combined.sink_parquet(f"{output_dir}/avg_pheno_per_var_371olink_EUR_corrected90pcs.parquet", engine='streaming')

In [2]:
ch_wgs = pl.scan_parquet("/home/dnanexus/data_dir/olink_appv_chunks_EUR/variant_pheno_chunk9.parquet")
ch_wgs.head().collect()

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:3672476:CGGCCCCA:C""","""ENSG00000019186_olink""",1,-1.209683,null
"""chr1:3689403:C:G""","""ENSG00000019186_olink""",2,0.758827,0.727729
"""chr1:3640236:C:T""","""ENSG00000019186_olink""",5,-0.642613,1.180599
"""chr1:3645018:A:G""","""ENSG00000161944_olink""",12,-0.715548,1.401296
"""chr1:3682246:C:T""","""ENSG00000161944_olink""",21,-0.153775,0.802043


In [4]:
odf = pl.scan_parquet(f"/home/dnanexus/data_dir/olink_appv_chunks_EUR_smarter/avg_pheno_per_var_371olink_EUR_corrected90pcs.parquet")
odf.head().collect()

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:3664499:C:T""","""ENSG00000134352_olink""",3,-0.272098,1.556659
"""chr1:3690579:C:T""","""ENSG00000134352_olink""",4,-0.183986,0.678737
"""chr1:3663475:G:T""","""ENSG00000134352_olink""",3,0.647399,1.883861
"""chr1:3706937:G:A""","""ENSG00000134352_olink""",1,-0.580956,null
"""chr1:3678411:G:A""","""ENSG00000134352_olink""",2,-0.301727,null


In [6]:
ch_wgs.join(odf.head(), on=['id', 'phenotype'], how='semi').collect()

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:3663475:G:T""","""ENSG00000134352_olink""",3,0.647399,1.883861
"""chr1:3678411:G:A""","""ENSG00000134352_olink""",2,-0.301727,null
"""chr1:3706937:G:A""","""ENSG00000134352_olink""",1,-0.580956,null
"""chr1:3690579:C:T""","""ENSG00000134352_olink""",4,-0.183986,0.678737
"""chr1:3664499:C:T""","""ENSG00000134352_olink""",3,-0.272098,1.556659
